# 04b2 — BiLSTM tuning (fix overfit)

**Problem from `04b`:** dev `f1_macro` peaks at epoch 2 (0.5999) then degrades
as `dev_loss` rises 0.78 → 1.18 — classic embedding overfit. Below LR
champion (0.6183).

**Hypotheses tested in 4 experiments:**

| # | Hypothesis | Change |
|---|---|---|
| 1 | Embedding (76% of params) memorises too fast | Freeze for 5 epochs, then unfreeze at lr=5e-5 |
| 2 | lr=1e-3 too aggressive for pretrained PhoW2V | lr=5e-4 + weight_decay=1e-3 |
| 3 | Regularisation too weak | LSTM dropout 0.5, head dropout 0.6, weight_decay=1e-3 |
| 4 | Filter dropped short toxic samples | min_length=2 instead of 3 (+ Exp 2 hyperparams) |

**Acceptance:** any version reaching dev `f1_macro ≥ 0.62` (beats LR
champion at 0.6183) wins. If none → log limitation, move to PhoBERT.

**Skip rule:** if Exp 1 ≥ 0.63, Exp 3 and Exp 4 are skipped to save time.

**Outputs:**

* `models/dl/bilstm_v2_best.pt`               — winning checkpoint
* `results/bilstm_tuning_log.json`            — full 4-experiment history
* `results/figures/17_bilstm_tuning_curves.png`
* `report/week4_phase1_summary.md`            — updated with new table


In [ ]:
# ── Setup: autoreload, paths, seed ──────────────────────────────────────
%load_ext autoreload
%autoreload 2

import sys
for _k in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[_k]

import os, json, time, copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path = [str(ROOT)] + [p for p in sys.path if p != str(ROOT)]

from configs.config import PATHS, COLUMNS, LABEL_MAP
from src.utils import set_seed, get_device

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
device = get_device()

DL_DIR        = ROOT / "models" / "dl"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR   = ROOT / "results"
FIG_DIR       = RESULTS_DIR / "figures"
REPORT_DIR    = ROOT / "report"
for d in (DL_DIR, RESULTS_DIR, FIG_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

TEXT, LABEL = COLUMNS["text"], COLUMNS["label"]

# Carried over from 04b, kept identical so comparison is apples-to-apples.
MAX_LEN        = 64
BATCH_SIZE     = 128
EVAL_BATCH     = 256
NUM_WORKERS    = 0
GRAD_CLIP      = 1.0
MAX_EPOCHS     = 10
ES_PATIENCE    = 3
SCHED_PATIENCE = 2

# If Exp 1 clears this, Exp 3 and Exp 4 are skipped.
EARLY_EXIT_F1  = 0.63

EMBEDDING_CHOICE      = "phow2v_300d"
EMBEDDING_MATRIX_PATH = DL_DIR / f"embedding_matrix_{EMBEDDING_CHOICE}.pt"

print(f"Device : {device}")
print(f"Seed   : {RANDOM_STATE}")


## 1. Load shared artefacts + datasets

Build two filtered training datasets up-front:

* `train_ds_m3` (min_length=3) — same filter as 04b, used by Exp 1-3
* `train_ds_m2` (min_length=2) — relaxed filter, used by Exp 4

Dev set stays unfiltered for all experiments — fair eval.

In [ ]:
from src.dataset_dl import Vocab, ViHSDDataset, FilteredViHSDDataset, collate_fn_bilstm

# vocab + embedding + class weights ───────────────────────────────────
vocab = Vocab.load(DL_DIR / "vocab.pkl")
_loaded = torch.load(EMBEDDING_MATRIX_PATH, weights_only=True)
emb_matrix = _loaded["embedding"] if isinstance(_loaded, dict) else _loaded
EMBEDDING_DIM = int(emb_matrix.shape[1])
class_weights = torch.load(DL_DIR / "class_weights.pt", weights_only=True)

train_df = pd.read_csv(PROCESSED_DIR / "train_cleaned.csv")
dev_df   = pd.read_csv(PROCESSED_DIR / "dev_cleaned.csv")

print(f"vocab  : {len(vocab):,}  |  embedding: {tuple(emb_matrix.shape)}  |  class_weights: {class_weights.tolist()}")
print(f"train  : {len(train_df):,}  |  dev: {len(dev_df):,}")

# Raw BiLSTM datasets ─────────────────────────────────────────────────
train_ds_raw = ViHSDDataset(
    texts     = train_df["cleaned"].fillna("").tolist(),
    labels    = train_df[LABEL].tolist(),
    tokenizer = vocab,
    max_len   = MAX_LEN,
    mode      = "bilstm",
)
dev_ds = ViHSDDataset(
    texts     = dev_df["cleaned"].fillna("").tolist(),
    labels    = dev_df[LABEL].tolist(),
    tokenizer = vocab,
    max_len   = MAX_LEN,
    mode      = "bilstm",
)

print()
print("Filter @ min_length=3 (Exp 1/2/3 train):")
train_ds_m3 = FilteredViHSDDataset(train_ds_raw, min_length=3, verbose=True)
print()
print("Filter @ min_length=2 (Exp 4 train):")
train_ds_m2 = FilteredViHSDDataset(train_ds_raw, min_length=2, verbose=True)

# Baseline (04b) history for the comparison table at the bottom. ──────
BASELINE_LOG = RESULTS_DIR / "bilstm_training_log.json"
baseline_history = []
baseline_best_epoch = 2
baseline_best = {"f1_macro": 0.5999, "f1_offensive": 0.394, "f1_hate": 0.523, "f1_clean": 0.92}
baseline_overfit_gap = 0.40
if BASELINE_LOG.exists():
    with open(BASELINE_LOG, "r", encoding="utf-8") as f:
        bl = json.load(f)
    baseline_history = bl.get("history", [])
    baseline_best_epoch = bl.get("best_epoch", baseline_best_epoch)
    if baseline_history:
        bi = baseline_best_epoch - 1
        bi = max(0, min(bi, len(baseline_history) - 1))
        baseline_best = {
            "f1_macro":     baseline_history[bi]["dev_f1_macro"],
            "f1_clean":     baseline_history[bi]["dev_f1_clean"],
            "f1_offensive": baseline_history[bi]["dev_f1_off"],
            "f1_hate":      baseline_history[bi]["dev_f1_hate"],
        }
        devlosses = [r["dev_loss"] for r in baseline_history]
        baseline_overfit_gap = max(devlosses) - min(devlosses)
    print(f"\nBaseline (04b): epoch {baseline_best_epoch}  F1m={baseline_best['f1_macro']:.4f}  "
          f"overfit_gap={baseline_overfit_gap:.3f}")
else:
    print(f"\n(No {BASELINE_LOG.name} — using hardcoded baseline from spec.)")


## 2. Reusable experiment runner

`run_experiment()` handles:

* deterministic seed reset (so any kwarg variation reproduces)
* freeze/unfreeze warmup (Exp 1)
* same eval / scheduler / early-stop logic as `04b`
* returns a dict packed with everything the comparison table needs.

In [ ]:
from src.models_dl import BiLSTMClassifier, count_parameters
from sklearn.metrics import f1_score, accuracy_score


def _train_one_epoch(model, loader, criterion, optimizer, grad_clip):
    model.train()
    total_loss, total_n = 0.0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        lengths   = batch["lengths"]
        labels    = batch["labels"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, lengths)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n    += bs
    return total_loss / max(total_n, 1)


@torch.no_grad()
def _eval_one_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_n = 0.0, 0
    preds, labs = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        lengths   = batch["lengths"]
        labels    = batch["labels"].to(device, non_blocking=True)
        logits = model(input_ids, lengths)
        loss   = criterion(logits, labels)
        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n    += bs
        preds.append(logits.argmax(-1).cpu().numpy())
        labs.append(labels.cpu().numpy())
    y_pred = np.concatenate(preds)
    y_true = np.concatenate(labs)
    per_f = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    return {
        "loss":         total_loss / max(total_n, 1),
        "acc":          float(accuracy_score(y_true, y_pred)),
        "f1_macro":     float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_clean":     float(per_f[0]),
        "f1_offensive": float(per_f[1]),
        "f1_hate":      float(per_f[2]),
    }


def run_experiment(
    name: str,
    train_ds,
    model_kwargs: dict,
    lr: float,
    weight_decay: float,
    *,
    max_epochs: int = MAX_EPOCHS,
    freeze_warmup_epochs: int = 0,
    post_warmup_lr: float = None,
    es_patience: int = ES_PATIENCE,
    sched_patience: int = SCHED_PATIENCE,
):
    """Run one experiment. Returns a result dict."""
    print(f"\n══ {name} ══════════════════════════════════════════════════")
    print(f"  model_kwargs : {model_kwargs}")
    print(f"  lr={lr}  weight_decay={weight_decay}  "
          f"freeze_warmup={freeze_warmup_epochs}  post_warmup_lr={post_warmup_lr}")

    set_seed(RANDOM_STATE)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn_bilstm,
        num_workers=NUM_WORKERS, pin_memory=True,
        generator=torch.Generator().manual_seed(RANDOM_STATE),
    )
    dev_loader = DataLoader(
        dev_ds, batch_size=EVAL_BATCH, shuffle=False,
        collate_fn=collate_fn_bilstm,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    model     = BiLSTMClassifier(**model_kwargs).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=sched_patience, min_lr=1e-5,
    )
    param_info = count_parameters(model, verbose=False)

    history = []
    best_f1, best_epoch, best_state = -1.0, -1, None
    patience_counter = 0
    t_start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        # Warmup unfreeze transition (Exp 1)
        if freeze_warmup_epochs and epoch == freeze_warmup_epochs + 1:
            model.set_embedding_frozen(False)
            new_lr = post_warmup_lr if post_warmup_lr is not None else optimizer.param_groups[0]["lr"]
            for g in optimizer.param_groups:
                g["lr"] = new_lr
            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  [{name}] epoch {epoch}: unfroze embedding "
                  f"({n_train:,} trainable params now), lr → {new_lr}")

        ep_t0 = time.perf_counter()
        tr_loss = _train_one_epoch(model, train_loader, criterion, optimizer, GRAD_CLIP)
        dev     = _eval_one_epoch(model, dev_loader, criterion)
        ep_t    = time.perf_counter() - ep_t0
        cur_lr  = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch":         epoch,
            "train_loss":    float(tr_loss),
            "dev_loss":      dev["loss"],
            "dev_acc":       dev["acc"],
            "dev_f1_macro":  dev["f1_macro"],
            "dev_f1_clean":  dev["f1_clean"],
            "dev_f1_off":    dev["f1_offensive"],
            "dev_f1_hate":   dev["f1_hate"],
            "lr":            float(cur_lr),
            "epoch_time_s":  float(ep_t),
        })

        scheduler.step(dev["f1_macro"])

        improved = dev["f1_macro"] > best_f1
        if improved:
            best_f1, best_epoch = dev["f1_macro"], epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        star = " ★" if improved else ""
        print(f"  [{name}] ep {epoch:>2d}/{max_epochs}  "
              f"train_loss={tr_loss:.4f}  dev_loss={dev['loss']:.4f}  "
              f"F1m={dev['f1_macro']:.4f}  "
              f"(CL={dev['f1_clean']:.3f} OF={dev['f1_offensive']:.3f} HA={dev['f1_hate']:.3f})  "
              f"lr={cur_lr:.1e}  t={ep_t:4.1f}s{star}")

        if patience_counter >= es_patience:
            print(f"  [{name}] ⏹ early stop at epoch {epoch}")
            break

    elapsed = time.perf_counter() - t_start
    bi      = best_epoch - 1
    best_row = history[bi] if 0 <= bi < len(history) else {}
    devlosses = [r["dev_loss"] for r in history]
    overfit_gap = float(max(devlosses) - min(devlosses)) if devlosses else 0.0

    result = {
        "name":             name,
        "history":          history,
        "best_epoch":       best_epoch,
        "best_f1_macro":    float(best_f1),
        "best_f1_clean":    float(best_row.get("dev_f1_clean", 0.0)),
        "best_f1_off":      float(best_row.get("dev_f1_off",   0.0)),
        "best_f1_hate":     float(best_row.get("dev_f1_hate",  0.0)),
        "overfit_gap":      overfit_gap,
        "elapsed_s":        elapsed,
        "state_dict":       best_state,
        "model_kwargs":     model_kwargs,
        "lr":               lr,
        "weight_decay":     weight_decay,
        "freeze_warmup_epochs": freeze_warmup_epochs,
        "post_warmup_lr":   post_warmup_lr,
        "param_info":       param_info,
        "min_length":       train_ds.min_length,
    }
    print(f"  [{name}] DONE: F1m={best_f1:.4f}  best_ep={best_epoch}  "
          f"gap={overfit_gap:.3f}  {elapsed:.0f}s")
    return result


# Bookkeeping for skip-rule.
results = []
skip_remaining = False


## 3. Exp 1 — Freeze warmup (5 epochs frozen, then unfreeze at lr=5e-5)

Cuts trainable params from ~3.5M → ~0.8M for the first 5 epochs so the
LSTM + head can settle before the (large) embedding layer gets to memorise.

In [ ]:
exp1 = run_experiment(
    name         = "Exp1_freeze_warmup",
    train_ds     = train_ds_m3,
    model_kwargs = dict(
        vocab_size            = len(vocab),
        embedding_dim         = EMBEDDING_DIM,
        hidden_dim            = 128,
        num_layers            = 2,
        num_classes           = 3,
        dropout               = 0.3,
        pretrained_embeddings = emb_matrix,
        freeze_embeddings     = True,    # frozen for warmup phase
        padding_idx           = 0,
        head_dropout          = 0.5,
    ),
    lr                   = 1e-3,
    weight_decay         = 1e-5,
    freeze_warmup_epochs = 5,
    post_warmup_lr       = 5e-5,
    max_epochs           = 10,
)
results.append(exp1)

if exp1["best_f1_macro"] > EARLY_EXIT_F1:
    skip_remaining = True
    print(f"\n✓ Exp 1 cleared {EARLY_EXIT_F1} → will skip Exp 3 and Exp 4.")


## 4. Exp 2 — Lower lr (5e-4) + stronger weight decay (1e-3)

Hypothesis: PhoW2V embeddings are already in a sensible region; lr=1e-3
pushes them out of that region in 1-2 epochs. Halve the lr, add real
L2 regularisation.

In [ ]:
exp2 = run_experiment(
    name         = "Exp2_low_lr",
    train_ds     = train_ds_m3,
    model_kwargs = dict(
        vocab_size            = len(vocab),
        embedding_dim         = EMBEDDING_DIM,
        hidden_dim            = 128,
        num_layers            = 2,
        num_classes           = 3,
        dropout               = 0.3,
        pretrained_embeddings = emb_matrix,
        freeze_embeddings     = False,
        padding_idx           = 0,
        head_dropout          = 0.5,
    ),
    lr           = 5e-4,
    weight_decay = 1e-3,
    max_epochs   = 10,
)
results.append(exp2)


## 5. Exp 3 — Stronger dropout + weight decay

Bigger dropout on both LSTM and head. lr stays at 1e-3 (so this is
strictly a regularisation lever, not an lr lever — clean attribution
when compared to Exp 2).

In [ ]:
if skip_remaining:
    print("⏭ Skipping Exp 3 — Exp 1 already cleared the threshold.")
    exp3 = None
else:
    exp3 = run_experiment(
        name         = "Exp3_strong_reg",
        train_ds     = train_ds_m3,
        model_kwargs = dict(
            vocab_size            = len(vocab),
            embedding_dim         = EMBEDDING_DIM,
            hidden_dim            = 128,
            num_layers            = 2,
            num_classes           = 3,
            dropout               = 0.5,       # LSTM inter-layer dropout up
            pretrained_embeddings = emb_matrix,
            freeze_embeddings     = False,
            padding_idx           = 0,
            head_dropout          = 0.6,       # head dropout up
        ),
        lr           = 1e-3,
        weight_decay = 1e-3,
        max_epochs   = 10,
    )
    results.append(exp3)


## 6. Exp 4 — Relax filter (min_length=2) + Exp 2 hyperparams

`04a` E1 said length≤2 is 11% of train. Most of those are short toxic
slurs ("dm", "vcl", etc.) — filtering them out might actively hurt the
OFFENSIVE / HATE recall. This experiment keeps them and combines with
the best-so-far regularisation regime (Exp 2).

In [ ]:
if skip_remaining:
    print("⏭ Skipping Exp 4 — Exp 1 already cleared the threshold.")
    exp4 = None
else:
    exp4 = run_experiment(
        name         = "Exp4_relax_filter",
        train_ds     = train_ds_m2,             # min_length=2 instead of 3
        model_kwargs = dict(
            vocab_size            = len(vocab),
            embedding_dim         = EMBEDDING_DIM,
            hidden_dim            = 128,
            num_layers            = 2,
            num_classes           = 3,
            dropout               = 0.3,
            pretrained_embeddings = emb_matrix,
            freeze_embeddings     = False,
            padding_idx           = 0,
            head_dropout          = 0.5,
        ),
        lr           = 5e-4,                    # match Exp 2
        weight_decay = 1e-3,                    # match Exp 2
        max_epochs   = 10,
    )
    results.append(exp4)


## 7. Comparison, verdict, save winner, update summary

Build the 5-version comparison table. Best dev `f1_macro` wins — save
its state_dict to `models/dl/bilstm_v2_best.pt`. Plot all training
curves overlaid. Update `report/week4_phase1_summary.md` with the new
results.

In [ ]:
# Build baseline + experiment rows for the comparison table.
def _row(name, f1m, f1c, f1o, f1h, best_ep, gap, params=None):
    return {
        "version":      name,
        "F1_macro":     f1m,
        "F1_CLEAN":     f1c,
        "F1_OFF":       f1o,
        "F1_HATE":      f1h,
        "best_epoch":   best_ep,
        "overfit_gap":  gap,
        "params":       params,
    }

baseline_params = None  # filled in from 04b log if present
if BASELINE_LOG.exists():
    with open(BASELINE_LOG, "r", encoding="utf-8") as f:
        bl = json.load(f)
    baseline_params = bl.get("param_info", {}).get("trainable")

rows = [_row(
    "Baseline (04b)",
    baseline_best["f1_macro"],
    baseline_best["f1_clean"],
    baseline_best["f1_offensive"],
    baseline_best["f1_hate"],
    baseline_best_epoch,
    baseline_overfit_gap,
    baseline_params,
)]
for r in results:
    rows.append(_row(
        r["name"],
        r["best_f1_macro"],
        r["best_f1_clean"],
        r["best_f1_off"],
        r["best_f1_hate"],
        r["best_epoch"],
        r["overfit_gap"],
        r["param_info"]["trainable"],
    ))

compare_df = pd.DataFrame(rows)
print("Comparison — 5 versions:\n")
print(compare_df.to_string(
    index=False,
    formatters={
        "F1_macro":    "{:.4f}".format,
        "F1_CLEAN":    "{:.4f}".format,
        "F1_OFF":      "{:.4f}".format,
        "F1_HATE":     "{:.4f}".format,
        "overfit_gap": "{:.3f}".format,
        "params":      lambda x: f"{x:,.0f}" if x is not None else "—",
    }))

# ── Pick winner among the experiments (not the baseline).
winner = max(results, key=lambda r: r["best_f1_macro"])
print(f"\n★ WINNER: {winner['name']}  F1m={winner['best_f1_macro']:.4f}  "
      f"(Δ vs baseline {winner['best_f1_macro']-baseline_best['f1_macro']:+.4f})")

# ── Verdict vs LR champion.
LR_F1 = 0.6183
beats_lr = winner["best_f1_macro"] >= LR_F1
beats_target = winner["best_f1_macro"] >= 0.62
print(f"  vs LR champion ({LR_F1}): {'✓ BEATS' if beats_lr else '✗ below'}  "
      f"(Δ {winner['best_f1_macro']-LR_F1:+.4f})")
print(f"  vs target (0.62)       : {'✓ MET' if beats_target else '✗ short'}")

# ── Save winning checkpoint.
V2_CKPT = DL_DIR / "bilstm_v2_best.pt"
torch.save({
    "experiment":         winner["name"],
    "best_epoch":         winner["best_epoch"],
    "model_state_dict":   winner["state_dict"],
    "metrics": {
        "f1_macro":     winner["best_f1_macro"],
        "f1_clean":     winner["best_f1_clean"],
        "f1_offensive": winner["best_f1_off"],
        "f1_hate":      winner["best_f1_hate"],
    },
    "config": {
        **winner["model_kwargs"],
        "max_len":              MAX_LEN,
        "min_length":           winner["min_length"],
        "batch_size":           BATCH_SIZE,
        "lr":                   winner["lr"],
        "weight_decay":         winner["weight_decay"],
        "grad_clip":            GRAD_CLIP,
        "freeze_warmup_epochs": winner["freeze_warmup_epochs"],
        "post_warmup_lr":       winner["post_warmup_lr"],
        "embedding_choice":     EMBEDDING_CHOICE,
    },
    "param_info":         winner["param_info"],
    "overfit_gap":        winner["overfit_gap"],
}, V2_CKPT)
# Strip state_dict from the JSON sibling (state_dict is the .pt file).
print(f"\n✓ saved → {V2_CKPT}")

# ── Persist full tuning log.
log_results = []
for r in results:
    rr = {k: v for k, v in r.items() if k != "state_dict"}
    # Cast model_kwargs values that aren't JSON-serialisable (the tensor).
    rr["model_kwargs"] = {k: (v if not isinstance(v, torch.Tensor) else f"<tensor {tuple(v.shape)}>")
                          for k, v in rr["model_kwargs"].items()}
    log_results.append(rr)
with open(RESULTS_DIR / "bilstm_tuning_log.json", "w", encoding="utf-8") as f:
    json.dump({
        "baseline": {
            "best_epoch": baseline_best_epoch,
            "best":       baseline_best,
            "overfit_gap": baseline_overfit_gap,
        },
        "experiments": log_results,
        "winner":      winner["name"],
        "winner_f1_macro": winner["best_f1_macro"],
    }, f, indent=2)
print(f"✓ saved → {RESULTS_DIR / 'bilstm_tuning_log.json'}")

# ── Plot: overlay dev_f1_macro + dev_loss across all 5 versions.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
colors = {"Baseline (04b)": "#888888"}
exp_palette = ["#4C78A8", "#54A24B", "#F58518", "#E45756"]
for r in results:
    colors[r["name"]] = exp_palette.pop(0)

# Baseline first (background).
if baseline_history:
    bh = pd.DataFrame(baseline_history)
    axes[0].plot(bh["epoch"], bh["dev_f1_macro"], "o--", color=colors["Baseline (04b)"], alpha=0.7, label="Baseline (04b)")
    axes[1].plot(bh["epoch"], bh["dev_loss"],     "o--", color=colors["Baseline (04b)"], alpha=0.7, label="Baseline (04b)")

# Experiments.
for r in results:
    h = pd.DataFrame(r["history"])
    c = colors[r["name"]]
    axes[0].plot(h["epoch"], h["dev_f1_macro"], "o-", color=c, label=r["name"])
    axes[1].plot(h["epoch"], h["dev_loss"],     "o-", color=c, label=r["name"])
    # mark best epoch with a star
    axes[0].plot([r["best_epoch"]], [r["best_f1_macro"]], marker="*", color=c, markersize=14, markeredgecolor="white")

axes[0].axhline(LR_F1, color="#666", linestyle=":", label=f"LR champion ({LR_F1})")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("dev F1 macro"); axes[0].set_title("Dev F1 macro")
axes[0].legend(loc="lower right", fontsize=8); axes[0].grid(alpha=0.3)

axes[1].set_xlabel("epoch"); axes[1].set_ylabel("dev loss"); axes[1].set_title("Dev loss (overfit signal)")
axes[1].legend(loc="upper left", fontsize=8); axes[1].grid(alpha=0.3)

fig.suptitle(f"BiLSTM tuning — winner: {winner['name']} (F1m={winner['best_f1_macro']:.4f})",
             y=1.02, fontsize=13)
fig_path = FIG_DIR / "17_bilstm_tuning_curves.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")

# ── Update report/week4_phase1_summary.md (append a tuning section).
SUMMARY = REPORT_DIR / "week4_phase1_summary.md"
existing = SUMMARY.read_text(encoding="utf-8") if SUMMARY.exists() else ""

lines = []
lines.append("")
lines.append("---")
lines.append("")
lines.append("## 8. Overfit fix (Notebook 04b2 — auto-appended)")
lines.append("")
lines.append(f"_Auto-appended by `04b2_bilstm_tuning.ipynb`. The baseline above ran with lr=1e-3 and "
             f"peaked at epoch {baseline_best_epoch} (F1m={baseline_best['f1_macro']:.4f}) before dev_loss "
             f"rose by {baseline_overfit_gap:.2f} — classic embedding overfit._")
lines.append("")
lines.append("### 8.1 Comparison")
lines.append("")
lines.append("| Version | F1 macro | F1 CLEAN | F1 OFF | F1 HATE | Best epoch | Overfit gap |")
lines.append("|---|---|---|---|---|---|---|")
for r in rows:
    lines.append(f"| {r['version']} "
                 f"| {r['F1_macro']:.4f} "
                 f"| {r['F1_CLEAN']:.4f} "
                 f"| {r['F1_OFF']:.4f} "
                 f"| {r['F1_HATE']:.4f} "
                 f"| {r['best_epoch']} "
                 f"| {r['overfit_gap']:.3f} |")
lines.append("")
lines.append("### 8.2 Verdict")
lines.append("")
lines.append(f"- **Winner:** `{winner['name']}` — dev F1_macro = {winner['best_f1_macro']:.4f} "
             f"(Δ vs baseline {winner['best_f1_macro']-baseline_best['f1_macro']:+.4f}).")
if beats_lr:
    lines.append(f"- ✓ Beats LR champion ({LR_F1}) by {winner['best_f1_macro']-LR_F1:+.4f}. "
                 f"Use `models/dl/bilstm_v2_best.pt` as the BiLSTM artefact going into Phase 2.")
else:
    lines.append(f"- ⚠ Best tuned BiLSTM ({winner['best_f1_macro']:.4f}) still trails LR champion ({LR_F1}). "
                 f"Static PhoW2V embeddings likely cap out here; PhoBERT (Phase 2) should clear "
                 f"this comfortably given its contextual embeddings + transformer attention.")
lines.append("")
lines.append("### 8.3 What worked / what didn't")
lines.append("")
# Per-experiment notes
deltas = [(r["name"], r["best_f1_macro"] - baseline_best["f1_macro"]) for r in results]
deltas.sort(key=lambda x: -x[1])
for name, d in deltas:
    tag = "✓" if d > 0 else "✗"
    lines.append(f"- {tag} `{name}` — Δ F1m vs baseline: {d:+.4f}")
lines.append("")
lines.append(f"### 8.4 Artefacts")
lines.append("")
lines.append(f"- `models/dl/bilstm_v2_best.pt` — winning checkpoint ({winner['name']})")
lines.append(f"- `results/bilstm_tuning_log.json`")
lines.append(f"- `results/figures/17_bilstm_tuning_curves.png`")

# If a previous tuning section exists from a past run, strip it before appending.
SEP = "## 8. Overfit fix"
if SEP in existing:
    existing = existing.split(SEP)[0].rstrip()
    # Remove the trailing '---' separator we added before that section.
    if existing.endswith("---"):
        existing = existing[:-3].rstrip()

SUMMARY.write_text(existing.rstrip() + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
print(f"✓ updated → {SUMMARY}")
print()
print(SUMMARY.read_text(encoding="utf-8").split(SEP)[-1] if SEP in SUMMARY.read_text(encoding="utf-8") else "(see file)")
